# publish

> Automate the nbdev edit->ship loop over the git/gh CLIs: nbdev-clean/export, a reviewed commit, push, wait for CI, and (on a feature branch) open and merge a PR to main. One confirm gate doubles as an interactive prompt and an agent read-and-authorize handoff. Only the 'github' destination is wired up so far (pypi/conda later).

In [ ]:
#| default_exp publish

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import subprocess, sys, time
from pathlib import Path

In [ ]:
#| export
def _run(cmd:list, cwd:str=None, capture:bool=True) -> subprocess.CompletedProcess:
    "Run `cmd` (an argv list, never a shell string -- no quoting/injection surface). `capture=False` lets it stream straight to the terminal (used for `gh run watch`, whose live progress we want to see). cwd defaults to the current directory."
    return subprocess.run(cmd, cwd=cwd, capture_output=capture, text=True)

def _git_out(args:list, cwd:str=None) -> str:
    "Stripped stdout of a `git` command, or '' if it failed -- for the small read-only queries (branch, HEAD sha, status)."
    r = _run(['git', *args], cwd=cwd)
    return r.stdout.strip() if r.returncode == 0 else ''

def _exe(name:str) -> str:
    "Resolve a console script (nbdev-clean/nbdev-export) next to the running interpreter if it's there, else fall back to bare PATH lookup -- so publish uses the same venv's nbdev as the project it's publishing, not whatever happens to be first on PATH."
    p = Path(sys.executable).parent / name
    return str(p) if p.exists() else name

In [ ]:
#| export
def _confirm(status:str, yes:bool) -> bool:
    "The proceed gate, shown after nbdev-clean/export so the review reflects generated changes too. Prints `status` (git status --short, so untracked files show as '??' and can't be silently missed by the later `git add -u`), then decides: `yes=True` proceeds outright (the -y override); on an interactive TTY it prompts 'Okay to proceed? [Y/n]'; on a non-TTY (a background run, an MCP/agent call) it returns False WITHOUT prompting -- the caller is meant to read the printed status and, if it looks right, re-invoke with yes=True. That single gate is thus both a human confirmation and an agent's read-and-authorize handoff."
    print(status or '(working tree clean)')
    if yes: return True
    if not sys.stdin.isatty():
        print("\n[non-interactive] Review the status above; re-run with yes=True (CLI: -y/--yes) to proceed.",
              file=sys.stderr)
        return False
    try: ans = input('Okay to proceed? [Y/n] ').strip().lower()
    except EOFError: return False
    return ans in ('', 'y', 'yes')

In [ ]:
#| export
def _wait_for_ci(sha:str, # the pushed commit whose CI run to wait on
                 branch:str, # the branch it was pushed to (used to locate the run)
                 cwd:str=None, # repo directory
                 appear_tries:int=30, # how many times to poll for the run to register before giving up
                 appear_delay:float=2.0 # seconds between those polls
                 ) -> 'bool|None':
    "Block until the GitHub Actions run triggered by `sha` finishes, via the `gh` CLI. Returns True (passed), False (failed), or None (gh unavailable, or no run ever registered for this commit). A run often doesn't appear the instant a push returns, so first poll `gh run list` for one whose headSha matches, then `gh run watch --exit-status` (which streams live progress and exits nonzero iff the run concluded in failure)."
    if _run(['gh', '--version'], cwd=cwd).returncode != 0:
        print('gh CLI not found -- skipping CI wait.', file=sys.stderr); return None
    rid = None
    for _ in range(appear_tries):
        r = _run(['gh', 'run', 'list', '--branch', branch, '--limit', '20',
                  '--json', 'databaseId,headSha', '--jq',
                  f'[.[]|select(.headSha=="{sha}")][0].databaseId'], cwd=cwd)
        rid = r.stdout.strip()
        if rid and rid != 'null': break
        rid = None
        time.sleep(appear_delay)
    if not rid:
        print(f'No CI run found for {sha[:8]} after ~{int(appear_tries*appear_delay)}s -- skipping CI wait.',
              file=sys.stderr)
        return None
    print(f'Watching CI run {rid} ...')
    return _run(['gh', 'run', 'watch', rid, '--exit-status'], cwd=cwd, capture=False).returncode == 0

In [ ]:
#| export
def publish(msg:str=None, # commit message; required only if there are changes to commit (prompted for on an interactive TTY if omitted). Ignored when the branch is already committed and just being shipped.
            dests:list=None, # publish destinations; only 'github' is implemented so far (pypi/conda later). Defaults to ['github'].
            merge:bool=True, # when on a non-main branch and CI passes, open a PR to main and merge it
            wait_ci:bool=True, # after pushing, wait for GitHub Actions to finish (forced on when a merge is needed -- a merge can't be gated without it)
            merge_method:str='merge', # how gh merges the PR: 'merge' | 'squash' | 'rebase'
            yes:bool=False, # skip the proceed prompt (the -y override); on a non-TTY without it, publish stops after showing git status so the caller can review and re-invoke
            cwd:str=None # repo directory (default: current directory)
            ) -> str:
    "Automate the edit->ship loop for an nbdev project. In order: nbdev-clean, nbdev-export, show `git status` and gate on it (see _confirm -- interactive prompt, or read-and-authorize for an agent), git add -u, commit (skipped if nothing is staged), push; then for a 'github' dest wait for CI (see _wait_for_ci), and if it passes while you're on a non-main branch with merge=True, open a PR to main and merge it (merge_method), then leave you on an updated local main. Stops at the first failing step and returns a summary. Only tracked files are staged (git add -u) -- new/untracked files show in the status review but aren't auto-added. If nothing is staged but the branch already has commits to ship, the commit step is skipped and it still pushes/CIs/merges."
    dests = dests if dests is not None else ['github']
    if merge_method not in ('merge', 'squash', 'rebase'):
        return f"merge_method must be 'merge'|'squash'|'rebase', got {merge_method!r}."
    if 'github' not in dests:
        return f"Nothing to do -- only 'github' is implemented so far, got dests={dests}."

    log = []
    def say(m): print(m); log.append(m)

    # 1-2. regenerate .py from the notebooks, using this venv's nbdev (see _exe)
    for name in ('nbdev-clean', 'nbdev-export'):
        r = _run([_exe(name)], cwd=cwd)
        if r.returncode != 0:
            return f"{name} failed (rc={r.returncode}):\n{r.stderr or r.stdout}"
        say(f"[ok] {name}")

    # 3-4. show status and gate on it (may run clean/export twice across a non-TTY review re-run -- both idempotent)
    if not _confirm(_git_out(['status', '--short', '--branch'], cwd=cwd), yes):
        return "Stopped before committing (awaiting confirmation)."

    # 5. stage tracked changes only; commit iff something is actually staged
    if _run(['git', 'add', '-u'], cwd=cwd).returncode != 0:
        return "git add -u failed."
    if _run(['git', 'diff', '--cached', '--quiet'], cwd=cwd).returncode != 0:  # something staged
        if not msg:
            msg = input('Commit message: ').strip() if sys.stdin.isatty() else ''
            if not msg: return "Changes are staged but no commit message given (pass msg=..., or --msg)."
        r = _run(['git', 'commit', '-m', msg], cwd=cwd)
        if r.returncode != 0:
            return f"git commit failed:\n{r.stderr or r.stdout}"
        say(f"[ok] committed: {msg.splitlines()[0]}")
    else:
        say("[ok] nothing new to commit -- shipping already-committed work")

    branch = _git_out(['rev-parse', '--abbrev-ref', 'HEAD'], cwd=cwd)
    sha = _git_out(['rev-parse', 'HEAD'], cwd=cwd)

    # 7. push (-u origin HEAD works whether or not the branch already has an upstream)
    r = _run(['git', 'push', '-u', 'origin', 'HEAD'], cwd=cwd)
    if r.returncode != 0:
        return "\n".join(log) + f"\ngit push failed:\n{r.stderr or r.stdout}"
    say(f"[ok] pushed origin/{branch} ({sha[:8]})")

    # 8-9. CI
    on_main = branch in ('main', 'master')
    if not (wait_ci or (merge and not on_main)):
        return "\n".join(log) + "\nDone (CI wait skipped)."
    ci = _wait_for_ci(sha, branch, cwd=cwd)
    if ci is False:
        return "\n".join(log) + "\n[FAIL] CI failed -- stopping. Run `slmn check_ci` for the failure logs."
    if ci is None:
        note = "  (a PR merge needs a confirmed CI pass, so not merging)" if (merge and not on_main) else ""
        return "\n".join(log) + f"\n[??] CI status unknown -- stopping.{note}"
    say("[ok] CI passed")

    # 10. non-main + merge -> PR to main and merge it
    if on_main or not merge:
        return "\n".join(log) + "\nDone."
    title = (msg or _git_out(['log', '-1', '--pretty=%s'], cwd=cwd)).splitlines()[0]
    body = msg or _git_out(['log', '-1', '--pretty=%B'], cwd=cwd)
    cr = _run(['gh', 'pr', 'create', '--base', 'main', '--head', branch, '--title', title, '--body', body], cwd=cwd)
    out = cr.stderr + cr.stdout
    if cr.returncode != 0:
        if 'No commits between' in out:
            return "\n".join(log) + "\nNothing to ship: branch has no commits beyond main."
        if 'already exists' not in out:
            return "\n".join(log) + f"\ngh pr create failed:\n{out}"
    mr = _run(['gh', 'pr', 'merge', branch, f'--{merge_method}', '--delete-branch'], cwd=cwd)
    if mr.returncode != 0:
        return "\n".join(log) + f"\ngh pr merge failed:\n{mr.stderr or mr.stdout}"
    say(f"[ok] merged {branch} -> main ({merge_method}), branch deleted")
    _run(['git', 'checkout', 'main'], cwd=cwd)
    _run(['git', 'pull'], cwd=cwd)
    say("[ok] local now on updated main")
    return "\n".join(log) + "\nDone."

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()